In [2]:
# Cell 1: Environment setup for Colab-only execution
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path('/content/semester-project') if IN_COLAB else Path.cwd()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

# Keep all installs inside notebook runtime as requested.
REQS = [
    'datasets>=2.19.0',
    'transformers>=4.40.0',
    'speechbrain>=0.5.16',
    'pyannote.metrics>=3.2.1',
    'soundfile>=0.12.1',
    'librosa>=0.10.2',
    'tqdm>=4.66.0',
    'matplotlib>=3.8.0',
    'seaborn>=0.13.0',
    'rich>=13.7.0'
 ]

def pip_install(packages, force_reinstall=False):
    if isinstance(packages, tuple) and len(packages) == 1 and isinstance(packages[0], list):
        packages = packages[0]
    cmd = [sys.executable, '-m', 'pip', 'install', '-U']
    if force_reinstall:
        cmd.extend(['--force-reinstall', '--no-cache-dir'])
    cmd.extend(packages)
    print('Installing runtime packages...')
    subprocess.check_call(cmd)

# Stabilize numeric stack first to avoid NumPy/SciPy ABI mismatches in notebook kernels.
pip_install(['numpy==1.26.4', 'scipy==1.11.4', 'scikit-learn==1.4.2'], force_reinstall=True)
# SpeechBrain versions in this notebook expect the legacy use_auth_token argument.
pip_install(['huggingface_hub==0.25.2'], force_reinstall=True)
pip_install(REQS)

print('Environment ready.')
print('Working directory:', Path.cwd())

Installing runtime packages...
Installing runtime packages...
Installing runtime packages...
Environment ready.
Working directory: /content/semester-project


In [ ]:
# Cell 2: Core imports and global configuration
from dataclasses import dataclass, asdict
from typing import Any, Dict, Iterable, List, Optional, Tuple
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

from datasets import load_dataset
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics.pairwise import cosine_similarity

# Compatibility shim for runtimes where torchaudio misses this API expected by speechbrain.
import torchaudio
if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile']

# Compatibility shim for SpeechBrain calling hf_hub_download(use_auth_token=...).
import inspect
import huggingface_hub
if 'use_auth_token' not in inspect.signature(huggingface_hub.hf_hub_download).parameters:
    _hf_hub_download_orig = huggingface_hub.hf_hub_download
    def _hf_hub_download_compat(*args, use_auth_token=None, **kwargs):
        if use_auth_token is not None and 'token' not in kwargs:
            kwargs['token'] = use_auth_token
        return _hf_hub_download_orig(*args, **kwargs)
    huggingface_hub.hf_hub_download = _hf_hub_download_compat

from speechbrain.inference.speaker import EncoderClassifier

warnings.filterwarnings('ignore')

@dataclass
class PipelineConfig:
    hf_dataset_name: str = 'diarizers-community/ami'
    dataset_config: str = 'ihm'  # set to 'sdm' for SDM condition
    sample_rate: int = 16000
    chunk_seconds: float = 1.5
    chunk_hop_seconds: float = 0.75
    min_chunk_seconds: float = 1.0
    xvector_model_source: str = 'speechbrain/spkrec-xvect-voxceleb'
    output_dir: str = 'artifacts'
    # Full-scale run (no subset limits).
    max_files_train_backend: Optional[int] = None
    max_files_eval: Optional[int] = None
    ahc_linkage: str = 'average'
    ahc_plda_threshold: float = 0.35
    collar: float = 0.25
    skip_overlap: bool = False
    random_seed: int = 42

CFG = PipelineConfig()
np.random.seed(CFG.random_seed)

ART = Path(CFG.output_dir)
(ART / 'manifests').mkdir(parents=True, exist_ok=True)
(ART / 'embeddings').mkdir(parents=True, exist_ok=True)
(ART / 'scores').mkdir(parents=True, exist_ok=True)
(ART / 'rttm').mkdir(parents=True, exist_ok=True)
(ART / 'metrics').mkdir(parents=True, exist_ok=True)

print('Loaded config:')
print(json.dumps(asdict(CFG), indent=2))

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover


Loaded config:
{
  "hf_dataset_name": "diarizers-community/ami",
  "dataset_config": "ihm",
  "sample_rate": 16000,
  "chunk_seconds": 1.5,
  "chunk_hop_seconds": 0.75,
  "min_chunk_seconds": 1.0,
  "xvector_model_source": "speechbrain/spkrec-xvect-voxceleb",
  "output_dir": "artifacts",
  "max_files_train_backend": 2,
  "max_files_eval": 1,
  "ahc_linkage": "average",
  "ahc_plda_threshold": 0.35,
  "collar": 0.25,
  "skip_overlap": false,
  "random_seed": 42
}


In [4]:
# Cell 3: Dataset schema inspection and robust column discovery
ds = load_dataset(CFG.hf_dataset_name, CFG.dataset_config)
print('Splits:', list(ds.keys()))
for split_name, split_ds in ds.items():
    print(f'[{split_name}] rows={len(split_ds)} columns={split_ds.column_names}')

def guess_columns(split_ds):
    cols = split_ds.column_names
    audio_col = None
    candidate_audio = [c for c in cols if 'audio' in c.lower() or 'wave' in c.lower()]
    if candidate_audio:
        audio_col = candidate_audio[0]
    else:
        for c in cols:
            if isinstance(split_ds[0].get(c, None), dict) and 'array' in split_ds[0][c]:
                audio_col = c
                break

    # We look for a field containing speaker turn annotations.
    ann_candidates = [c for c in cols if any(k in c.lower() for k in ['annotation', 'segment', 'turn', 'rttm', 'speaker'])]
    return {'audio_col': audio_col, 'ann_candidates': ann_candidates, 'all_cols': cols}

schema_info = {s: guess_columns(ds[s]) for s in ds.keys()}
print(json.dumps(schema_info, indent=2, default=str))

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Splits: ['train', 'validation', 'test']
[train] rows=136 columns=['audio', 'timestamps_start', 'timestamps_end', 'speakers']
[validation] rows=18 columns=['audio', 'timestamps_start', 'timestamps_end', 'speakers']
[test] rows=16 columns=['audio', 'timestamps_start', 'timestamps_end', 'speakers']
{
  "train": {
    "audio_col": "audio",
    "ann_candidates": [
      "speakers"
    ],
    "all_cols": [
      "audio",
      "timestamps_start",
      "timestamps_end",
      "speakers"
    ]
  },
  "validation": {
    "audio_col": "audio",
    "ann_candidates": [
      "speakers"
    ],
    "all_cols": [
      "audio",
      "timestamps_start",
      "timestamps_end",
      "speakers"
    ]
  },
  "test": {
    "audio_col": "audio",
    "ann_candidates": [
      "speakers"
    ],
    "all_cols": [
      "audio",
      "timestamps_start",
      "timestamps_end",
      "speakers"
    ]
  }
}


In [5]:
# Cell 4: Utilities for annotation parsing and RTTM conversion
def ensure_audio_array(audio_obj, target_sr=16000):
    if audio_obj is None:
        raise ValueError('Audio object is None.')

    # Hugging Face datasets may return an AudioDecoder object in newer versions.
    if hasattr(audio_obj, 'get_all_samples'):
        decoded = audio_obj.get_all_samples()
        arr = getattr(decoded, 'data', decoded)
        sr = int(getattr(decoded, 'sample_rate', target_sr))
        if hasattr(arr, 'numpy'):
            arr = arr.numpy()
        arr = np.asarray(arr, dtype=np.float32)
    elif isinstance(audio_obj, dict) and 'array' in audio_obj:
        arr = np.asarray(audio_obj['array'], dtype=np.float32)
        sr = int(audio_obj.get('sampling_rate', target_sr))
    elif isinstance(audio_obj, (list, np.ndarray)):
        arr = np.asarray(audio_obj, dtype=np.float32)
        sr = target_sr
    elif isinstance(audio_obj, str):
        arr, sr = sf.read(audio_obj)
        arr = np.asarray(arr, dtype=np.float32)
    else:
        raise TypeError(f'Unsupported audio format: {type(audio_obj)}')

    if arr.ndim == 2:
        # Handle both [time, channels] and [channels, time].
        arr = arr.mean(axis=1) if arr.shape[0] >= arr.shape[1] else arr.mean(axis=0)
    if sr != target_sr:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    return arr.astype(np.float32), sr

def normalize_segments_from_row(row: Dict[str, Any], ann_candidates: List[str]) -> List[Dict[str, Any]]:
    segs = []
    for c in ann_candidates:
        v = row.get(c, None)
        if v is None:
            continue

        # Case A: direct list of segment dicts.
        if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
            for item in v:
                start = item.get('start', item.get('start_time', item.get('begin', None)))
                end = item.get('end', item.get('end_time', item.get('stop', None)))
                spk = item.get('speaker', item.get('speaker_id', item.get('label', None)))
                if start is not None and end is not None and spk is not None and end > start:
                    segs.append({'start': float(start), 'end': float(end), 'speaker': str(spk)})

        # Case B: dict of arrays, e.g., {'start': [...], 'end': [...], 'speaker': [...]}
        if isinstance(v, dict):
            starts = v.get('start', v.get('start_time', v.get('begin', [])))
            ends = v.get('end', v.get('end_time', v.get('stop', [])))
            spks = v.get('speaker', v.get('speaker_id', v.get('label', [])))
            n = min(len(starts), len(ends), len(spks)) if hasattr(starts, '__len__') else 0
            for i in range(n):
                st = float(starts[i])
                en = float(ends[i])
                if en > st:
                    segs.append({'start': st, 'end': en, 'speaker': str(spks[i])})

    # Fallback: scan row keys for parallel arrays.
    if not segs:
        keys = {k.lower(): k for k in row.keys()}
        start_k = next((keys[k] for k in keys if 'start' in k or 'begin' in k), None)
        end_k = next((keys[k] for k in keys if 'end' in k or 'stop' in k), None)
        spk_k = next((keys[k] for k in keys if 'speaker' in k or 'label' in k), None)
        if start_k and end_k and spk_k:
            starts, ends, spks = row[start_k], row[end_k], row[spk_k]
            if isinstance(starts, list) and isinstance(ends, list) and isinstance(spks, list):
                n = min(len(starts), len(ends), len(spks))
                for i in range(n):
                    st, en = float(starts[i]), float(ends[i])
                    if en > st:
                        segs.append({'start': st, 'end': en, 'speaker': str(spks[i])})

    segs = sorted(segs, key=lambda x: (x['start'], x['end']))
    return segs

def write_rttm(records: List[Dict[str, Any]], out_path: Path):
    with open(out_path, 'w', encoding='utf-8') as f:
        for r in records:
            dur = max(0.0, r['end'] - r['start'])
            line = f"SPEAKER {r['uri']} 1 {r['start']:.3f} {dur:.3f} <NA> <NA> {r['speaker']} <NA> <NA>\n"
            f.write(line)

In [6]:
# Cell 5: Manifest builder with train/dev/eval supervision
def get_recording_uri(row: Dict[str, Any]) -> str:
    for k in ['meeting_id', 'recording_id', 'session_id', 'uri', 'id']:
        if k in row and row[k] is not None:
            return str(row[k])
    return str(abs(hash(json.dumps({k: str(v) for k, v in row.items() if k != 'audio'}))))

def build_manifest(split_name: str, limit: Optional[int] = None) -> pd.DataFrame:
    split_ds = ds[split_name]
    info = schema_info[split_name]
    audio_col = info['audio_col']
    ann_candidates = info['ann_candidates']

    if audio_col is None:
        raise ValueError(f'No audio column found for split={split_name}. Columns={info["all_cols"]}')
    if not ann_candidates:
        raise ValueError(f'No annotation-like columns found for split={split_name}. Columns={info["all_cols"]}')

    rows = []
    n = len(split_ds) if limit is None else min(limit, len(split_ds))
    for idx in tqdm(range(n), desc=f'build_manifest:{split_name}'):
        row = split_ds[idx]
        uri = get_recording_uri(row)
        segs = normalize_segments_from_row(row, ann_candidates)
        if not segs:
            continue
        duration = max(s['end'] for s in segs)
        rows.append({
            'split': split_name,
            'idx': int(idx),
            'uri': uri,
            'audio_col': audio_col,
            'duration': float(duration),
            'segments': segs
        })

    manifest = pd.DataFrame(rows)
    print(f'{split_name}: kept {len(manifest)} recordings with usable supervision')
    return manifest

train_manifest = build_manifest('train', CFG.max_files_train_backend)
dev_manifest = build_manifest('validation' if 'validation' in ds else 'test', CFG.max_files_eval)
eval_split = 'test' if 'test' in ds else ('eval' if 'eval' in ds else list(ds.keys())[-1])
eval_manifest = build_manifest(eval_split, CFG.max_files_eval)

train_manifest.to_pickle(ART / 'manifests' / 'train_manifest.pkl')
dev_manifest.to_pickle(ART / 'manifests' / 'dev_manifest.pkl')
eval_manifest.to_pickle(ART / 'manifests' / 'eval_manifest.pkl')

print('Saved manifests:', ART / 'manifests')

build_manifest:train:   0%|          | 0/2 [00:00<?, ?it/s]

train: kept 2 recordings with usable supervision


build_manifest:validation:   0%|          | 0/1 [00:00<?, ?it/s]

validation: kept 1 recordings with usable supervision


build_manifest:test:   0%|          | 0/1 [00:00<?, ?it/s]

test: kept 1 recordings with usable supervision
Saved manifests: artifacts/manifests


In [7]:
# Cell 6: Build supervised chunks for x-vector and diarization back-end
def intersect(a0, a1, b0, b1):
    st = max(a0, b0)
    en = min(a1, b1)
    return (st, en) if en > st else None

def segment_speaker_overlap(chunk_st, chunk_en, ref_segments):
    # Returns speaker with max overlap in chunk, and overlap duration.
    best_spk, best_ov = None, 0.0
    for seg in ref_segments:
        x = intersect(chunk_st, chunk_en, seg['start'], seg['end'])
        if x is None:
            continue
        ov = x[1] - x[0]
        if ov > best_ov:
            best_ov = ov
            best_spk = seg['speaker']
    return best_spk, best_ov

def make_chunks(manifest: pd.DataFrame, training_mode: bool) -> pd.DataFrame:
    rows = []
    for _, rec in tqdm(manifest.iterrows(), total=len(manifest), desc='make_chunks'):
        split_name = rec['split']
        idx = int(rec['idx'])
        segs = rec['segments']

        row = ds[split_name][idx]
        wav, sr = ensure_audio_array(row[rec['audio_col']], target_sr=CFG.sample_rate)
        dur = len(wav) / sr

        win = CFG.chunk_seconds
        hop = CFG.chunk_hop_seconds
        n_steps = max(1, int(math.floor((dur - win) / hop)) + 1) if dur > win else 1

        for i in range(n_steps):
            st = i * hop
            en = min(dur, st + win)
            if en - st < CFG.min_chunk_seconds:
                continue
            s, e = int(st * sr), int(en * sr)
            chunk = wav[s:e]
            if len(chunk) < int(CFG.min_chunk_seconds * sr):
                continue

            spk, ov = segment_speaker_overlap(st, en, segs)
            if spk is None:
                continue

            if training_mode:
                purity = ov / max(1e-8, (en - st))
                if purity < 0.7:
                    # For robust x-vector and PLDA training, keep mostly single-speaker windows.
                    continue

            rows.append({
                'uri': rec['uri'],
                'split': split_name,
                'start': st,
                'end': en,
                'speaker': spk,
                'audio': chunk.astype(np.float32),
                'sr': sr
            })

    chunks = pd.DataFrame(rows)
    print(f'Chunk rows: {len(chunks)}')
    return chunks

train_chunks = make_chunks(train_manifest, training_mode=True)
dev_chunks = make_chunks(dev_manifest, training_mode=False)
eval_chunks = make_chunks(eval_manifest, training_mode=False)

train_chunks.to_pickle(ART / 'manifests' / 'train_chunks.pkl')
dev_chunks.to_pickle(ART / 'manifests' / 'dev_chunks.pkl')
eval_chunks.to_pickle(ART / 'manifests' / 'eval_chunks.pkl')

make_chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Chunk rows: 6057


make_chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk rows: 3805


make_chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk rows: 2158


In [8]:
# Cell 7: X-vector extraction (TDNN)
from huggingface_hub import snapshot_download

xvec_repo_dir = ART / 'models' / 'xvector_repo'
xvec_repo_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=CFG.xvector_model_source,
    local_dir=str(xvec_repo_dir),
    local_dir_use_symlinks=False
 )

# Some SpeechBrain checkpoints do not provide custom.py, but the loader expects it.
custom_py = xvec_repo_dir / 'custom.py'
if not custom_py.exists():
    custom_py.write_text('# Auto-generated compatibility stub for SpeechBrain loader\n', encoding='utf-8')

xvec_model = EncoderClassifier.from_hparams(
    source=str(xvec_repo_dir),
    hparams_file='hyperparams.yaml',
    pymodule_file='custom.py',
    savedir=str(ART / 'models' / 'xvector')
 )

def l2_normalize(v):
    n = np.linalg.norm(v) + 1e-12
    return v / n

def extract_embeddings(chunks: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    rows = []
    batch_audio = []
    batch_meta = []

    def flush_batch():
        nonlocal batch_audio, batch_meta, rows
        if not batch_audio:
            return
        max_len = max(x.shape[0] for x in batch_audio)
        padded = np.stack([np.pad(x, (0, max_len - len(x))) for x in batch_audio], axis=0)
        wavs = padded
        # speechbrain expects torch tensor; import local to avoid hard dependency before install cell
        import torch
        with torch.no_grad():
            embs = xvec_model.encode_batch(torch.tensor(wavs, dtype=torch.float32))
            embs = embs.squeeze(1).cpu().numpy()

        for meta, emb in zip(batch_meta, embs):
            rows.append({**meta, 'embedding': l2_normalize(emb.astype(np.float32))})

        batch_audio = []
        batch_meta = []

    for _, r in tqdm(chunks.iterrows(), total=len(chunks), desc='extract_embeddings'):
        batch_audio.append(r['audio'])
        batch_meta.append({
            'uri': r['uri'],
            'split': r['split'],
            'start': r['start'],
            'end': r['end'],
            'speaker': r['speaker']
        })
        if len(batch_audio) >= batch_size:
            flush_batch()
    flush_batch()

    out = pd.DataFrame(rows)
    print('Embeddings shape sample:', out.iloc[0]['embedding'].shape if len(out) else None)
    return out

train_emb = extract_embeddings(train_chunks)
dev_emb = extract_embeddings(dev_chunks)
eval_emb = extract_embeddings(eval_chunks)

train_emb.to_pickle(ART / 'embeddings' / 'train_emb.pkl')
dev_emb.to_pickle(ART / 'embeddings' / 'dev_emb.pkl')
eval_emb.to_pickle(ART / 'embeddings' / 'eval_emb.pkl')

print('Saved embeddings to', ART / 'embeddings')

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/semester-project/artifacts/models/xvector/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Using symlink found at '/content/semester-project/artifacts/models/xvector/custom.py'
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered parameter transfer hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load_if_possible
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in artifacts/models/xvector.
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/semester-project/artifacts/models/xvector/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_tra

extract_embeddings:   0%|          | 0/6057 [00:00<?, ?it/s]

Embeddings shape sample: (512,)


extract_embeddings:   0%|          | 0/3805 [00:00<?, ?it/s]

Embeddings shape sample: (512,)


extract_embeddings:   0%|          | 0/2158 [00:00<?, ?it/s]

Embeddings shape sample: (512,)
Saved embeddings to artifacts/embeddings


In [9]:
# Cell 8: PLDA backend fitting and pairwise scoring
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.covariance import EmpiricalCovariance

def fit_lda(train_df: pd.DataFrame, max_dim: int = 150):
    X = np.stack(train_df['embedding'].values)
    y = train_df['speaker'].astype(str).values
    n_classes = len(np.unique(y))
    out_dim = max(2, min(max_dim, X.shape[1], n_classes - 1))
    lda = LinearDiscriminantAnalysis(n_components=out_dim)
    lda.fit(X, y)
    return lda

def fit_simplified_plda(train_df: pd.DataFrame, lda_model=None):
    # Real-world approximation: LDA projection + between/within covariance model.
    X = np.stack(train_df['embedding'].values)
    y = train_df['speaker'].astype(str).values
    if lda_model is not None:
        X = lda_model.transform(X)

    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu

    spk_means = []
    Sw_acc = np.zeros((X.shape[1], X.shape[1]), dtype=np.float64)
    sw_count = 0

    for spk in np.unique(y):
        Xi = X[y == spk]
        mi = Xi.mean(axis=0, keepdims=True)
        spk_means.append(mi.squeeze(0))
        centered = Xi - mi
        Sw_acc += centered.T @ centered
        sw_count += max(1, Xi.shape[0] - 1)

    spk_means = np.stack(spk_means, axis=0)
    Sb = np.cov(spk_means.T, bias=False)
    Sw = Sw_acc / max(1, sw_count)

    # Regularization for numerical stability.
    eps = 1e-3
    Sw = Sw + eps * np.eye(Sw.shape[0])
    Sb = Sb + eps * np.eye(Sb.shape[0])

    invSw = np.linalg.inv(Sw)
    model = {'mu': mu.squeeze(0), 'Sw': Sw, 'Sb': Sb, 'invSw': invSw, 'lda': lda_model}
    return model

def plda_score_pair(x1, x2, model):
    # Symmetric quadratic form using between-speaker covariance as cross-term.
    x1 = x1 - model['mu']
    x2 = x2 - model['mu']
    A = model['invSw'] @ model['Sb'] @ model['invSw']
    return float(x1 @ A @ x2)

def transform_emb(E: np.ndarray, plda_model):
    lda = plda_model['lda']
    if lda is None:
        return E
    return lda.transform(E)

lda_model = fit_lda(train_emb, max_dim=150)
plda_model = fit_simplified_plda(train_emb, lda_model=lda_model)
print('PLDA backend trained. Dim:', plda_model['Sw'].shape[0])

PLDA backend trained. Dim: 7


In [10]:
# Cell 9: AHC diarization using PLDA similarity
def score_matrix_from_plda(emb_df: pd.DataFrame, plda_model) -> Tuple[np.ndarray, List[str], np.ndarray, np.ndarray]:
    E = np.stack(emb_df['embedding'].values)
    E = transform_emb(E, plda_model)
    n = E.shape[0]
    S = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        S[i, i] = 1.0
        for j in range(i + 1, n):
            s = plda_score_pair(E[i], E[j], plda_model)
            S[i, j] = s
            S[j, i] = s
    return S, emb_df['uri'].tolist(), emb_df['start'].values, emb_df['end'].values

def ahc_cluster_from_scores(S: np.ndarray, threshold: float, linkage_method: str = 'average') -> np.ndarray:
    # Convert similarity to distance while preserving rank ordering.
    S = S - S.min()
    if S.max() > 0:
        S = S / S.max()
    D = 1.0 - S
    np.fill_diagonal(D, 0.0)
    condensed = squareform(D, checks=False)
    Z = linkage(condensed, method=linkage_method)
    labels = fcluster(Z, t=threshold, criterion='distance')
    return labels.astype(int)

def diarize_one_recording_with_ahc(emb_df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    S, uris, starts, ends = score_matrix_from_plda(emb_df, plda_model)
    labels = ahc_cluster_from_scores(S, threshold=threshold, linkage_method=CFG.ahc_linkage)
    out = emb_df[['uri', 'start', 'end']].copy()
    out['speaker'] = [f'spk_{x:03d}' for x in labels]
    out = out.sort_values(['start', 'end']).reset_index(drop=True)
    return out

def diarize_manifest_ahc(embeddings_df: pd.DataFrame, threshold: float, out_rttm: Path):
    all_rows = []
    for uri, part in tqdm(embeddings_df.groupby('uri'), desc='AHC diarization'):
        hyp = diarize_one_recording_with_ahc(part.reset_index(drop=True), threshold=threshold)
        all_rows.extend(hyp.to_dict('records'))
    write_rttm(all_rows, out_rttm)
    return pd.DataFrame(all_rows)

ahc_dev_rttm = ART / 'rttm' / 'dev_ahc.rttm'
ahc_eval_rttm = ART / 'rttm' / 'eval_ahc.rttm'

ahc_dev_df = diarize_manifest_ahc(dev_emb, CFG.ahc_plda_threshold, ahc_dev_rttm)
ahc_eval_df = diarize_manifest_ahc(eval_emb, CFG.ahc_plda_threshold, ahc_eval_rttm)

print('Wrote AHC RTTM files:', ahc_dev_rttm, ahc_eval_rttm)

AHC diarization:   0%|          | 0/1 [00:00<?, ?it/s]

AHC diarization:   0%|          | 0/1 [00:00<?, ?it/s]

Wrote AHC RTTM files: artifacts/rttm/dev_ahc.rttm artifacts/rttm/eval_ahc.rttm


In [11]:
# Cell 10: DER and JER evaluation (pyannote.metrics)
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate

def build_reference_annotation(manifest_df: pd.DataFrame, uri: str) -> Annotation:
    ann = Annotation(uri=uri)
    rec = manifest_df[manifest_df['uri'] == uri]
    if len(rec) == 0:
        return ann
    segs = rec.iloc[0]['segments']
    for i, s in enumerate(segs):
        ann[Segment(float(s['start']), float(s['end']))] = str(s['speaker'])
    return ann

def build_hyp_annotation(hyp_df: pd.DataFrame, uri: str) -> Annotation:
    ann = Annotation(uri=uri)
    rec = hyp_df[hyp_df['uri'] == uri]
    for i, r in rec.iterrows():
        ann[Segment(float(r['start']), float(r['end']))] = str(r['speaker'])
    return ann

def evaluate_manifest(manifest_df: pd.DataFrame, hyp_df: pd.DataFrame, title: str):
    der_metric = DiarizationErrorRate(collar=CFG.collar, skip_overlap=CFG.skip_overlap)
    jer_metric = JaccardErrorRate(collar=CFG.collar, skip_overlap=CFG.skip_overlap)

    common_uris = sorted(set(manifest_df['uri']).intersection(set(hyp_df['uri'])))
    for uri in tqdm(common_uris, desc=f'eval:{title}'):
        ref = build_reference_annotation(manifest_df, uri)
        hyp = build_hyp_annotation(hyp_df, uri)
        der_metric(ref, hyp)
        jer_metric(ref, hyp)

    der = abs(der_metric)
    jer = abs(jer_metric)
    print(f'{title} DER={der:.4f} JER={jer:.4f}')
    return {'title': title, 'DER': der, 'JER': jer}

ahc_dev_metrics = evaluate_manifest(dev_manifest, ahc_dev_df, 'DEV AHC-PLDA')
ahc_eval_metrics = evaluate_manifest(eval_manifest, ahc_eval_df, 'EVAL AHC-PLDA')

pd.DataFrame([ahc_dev_metrics, ahc_eval_metrics]).to_csv(ART / 'metrics' / 'ahc_metrics.csv', index=False)

eval:DEV AHC-PLDA:   0%|          | 0/1 [00:00<?, ?it/s]

DEV AHC-PLDA DER=1.8477 JER=0.9761


eval:EVAL AHC-PLDA:   0%|          | 0/1 [00:00<?, ?it/s]

EVAL AHC-PLDA DER=1.7000 JER=0.9638


In [12]:
# Cell 11: Threshold sweep on dev for AHC+PLDA tuning
def threshold_sweep(dev_embeddings: pd.DataFrame, dev_manifest_df: pd.DataFrame, thr_values: Iterable[float]):
    results = []
    for thr in thr_values:
        hyp_rttm = ART / 'rttm' / f'dev_ahc_thr_{thr:.3f}.rttm'
        hyp_df = diarize_manifest_ahc(dev_embeddings, threshold=float(thr), out_rttm=hyp_rttm)
        m = evaluate_manifest(dev_manifest_df, hyp_df, f'DEV AHC thr={thr:.3f}')
        m['threshold'] = float(thr)
        results.append(m)
    out_df = pd.DataFrame(results).sort_values('DER').reset_index(drop=True)
    return out_df

thr_grid = np.linspace(0.25, 0.75, 11)
sweep_df = threshold_sweep(dev_emb, dev_manifest, thr_grid)
sweep_df.to_csv(ART / 'metrics' / 'ahc_threshold_sweep.csv', index=False)
best_thr = float(sweep_df.iloc[0]['threshold'])
print('Best dev threshold:', best_thr)
display(sweep_df.head(10))

# Re-run eval using best dev threshold for proper protocol
best_eval_rttm = ART / 'rttm' / 'eval_ahc_bestthr.rttm'
best_eval_df = diarize_manifest_ahc(eval_emb, threshold=best_thr, out_rttm=best_eval_rttm)
best_eval_metrics = evaluate_manifest(eval_manifest, best_eval_df, 'EVAL AHC-PLDA best-dev-threshold')

pd.DataFrame([best_eval_metrics]).to_csv(ART / 'metrics' / 'ahc_best_eval.csv', index=False)

AHC diarization:   0%|          | 0/1 [00:00<?, ?it/s]

eval:DEV AHC thr=0.250:   0%|          | 0/1 [00:00<?, ?it/s]

DEV AHC thr=0.250 DER=1.8616 JER=0.9920


AHC diarization:   0%|          | 0/1 [00:00<?, ?it/s]

eval:DEV AHC thr=0.300:   0%|          | 0/1 [00:00<?, ?it/s]

...

DEV AHC thr=0.750 DER=1.5540 JER=0.9160
Best dev threshold: 0.65


,title,DER,JER,threshold
0,DEV AHC thr=0.650,1.403515,0.687793,0.65
1,DEV AHC thr=0.700,1.553990,0.915957,0.70
2,DEV AHC thr=0.750,1.553990,0.915957,0.75
3,DEV AHC thr=0.600,1.612907,0.777095,0.60
4,DEV AHC thr=0.550,1.701048,0.826125,0.55
5,DEV AHC thr=0.500,1.759828,0.880566,0.50
6,DEV AHC thr=0.450,1.799691,0.922296,0.45
7,DEV AHC thr=0.400,1.829902,0.955933,0.40
8,DEV AHC thr=0.350,1.847707,0.976102,0.35
9,DEV AHC thr=0.300,1.856060,0.985603,0.30


AHC diarization:   0%|          | 0/1 [00:00<?, ?it/s]

eval:EVAL AHC-PLDA best-dev-threshold:   0%|          | 0/1 [00:00<?, ?it/s]

EVAL AHC-PLDA best-dev-threshold DER=1.2112 JER=0.7071


In [ ]:
# Cell 12: Official VBx setup and data preparation (inside notebook runtime)
import shutil
import subprocess

def run_cmd(cmd, cwd=None):
    print('RUN:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    vbx_log = ART / 'vbx_work' / 'last_cmd.log'
    vbx_log.parent.mkdir(parents=True, exist_ok=True)
    with open(vbx_log, 'w', encoding='utf-8') as lf:
        lf.write('STDOUT:\n')
        lf.write(proc.stdout or '')
        lf.write('\n\nSTDERR:\n')
        lf.write(proc.stderr or '')
    if proc.returncode != 0:
        stdout_tail = (proc.stdout or '')[-3000:]
        stderr_tail = (proc.stderr or '')[-3000:]
        if stdout_tail:
            print('--- STDOUT tail ---')
            print(stdout_tail)
        if stderr_tail:
            print('--- STDERR tail ---')
            print(stderr_tail)
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}. Full log: {vbx_log}')
    if proc.stdout:
        print(proc.stdout[-1500:])

# Extra dependencies required by official VBx scripts.
pip_install(['kaldi-io', 'fastcluster', 'h5py', 'onnxruntime'])

VBX_DIR = PROJECT_ROOT / 'VBx'
if not VBX_DIR.exists():
    run_cmd(['git', 'clone', 'https://github.com/BUTSpeechFIT/VBx.git', str(VBX_DIR)])

run_cmd([sys.executable, '-m', 'pip', 'install', '-e', str(VBX_DIR)])

# Official model assets are under VBx/VBx/models/...
vbx_model_root = VBX_DIR / 'VBx' / 'models' / 'ResNet101_16kHz'
if not vbx_model_root.exists():
    raise FileNotFoundError(
        f'VBx model directory not found at {vbx_model_root}. '
        'In Colab, ensure the repository includes model files or fetch the official model package.'
    )

vbx_work = ART / 'vbx_work'
(vbx_work / 'audios_16k').mkdir(parents=True, exist_ok=True)
(vbx_work / 'vad').mkdir(parents=True, exist_ok=True)
(vbx_work / 'ref_rttm').mkdir(parents=True, exist_ok=True)
(vbx_work / 'exp').mkdir(parents=True, exist_ok=True)

def write_oracle_vad_lab(ref_segments: List[Dict[str, Any]], out_lab: Path):
    # VBx predict.py accepts .lab speech activity intervals (start end speech).
    with open(out_lab, 'w', encoding='utf-8') as f:
        for s in ref_segments:
            f.write(f"{float(s['start']):.3f} {float(s['end']):.3f} speech\\n")

# Use full dev set for VBx runs (no cap).
selected_uris = sorted(dev_manifest['uri'].unique())
with open(vbx_work / 'list.txt', 'w', encoding='utf-8') as lf:
    for uri in selected_uris:
        rec = dev_manifest[dev_manifest['uri'] == uri].iloc[0]
        row = ds[rec['split']][int(rec['idx'])]
        wav, sr = ensure_audio_array(row[rec['audio_col']], target_sr=CFG.sample_rate)
        wav_path = vbx_work / 'audios_16k' / f'{uri}.wav'
        sf.write(wav_path, wav, sr)

        lab_path = vbx_work / 'vad' / f'{uri}.lab'
        write_oracle_vad_lab(rec['segments'], lab_path)

        ref_path = vbx_work / 'ref_rttm' / f'{uri}.rttm'
        ref_rows = [{'uri': uri, 'start': s['start'], 'end': s['end'], 'speaker': s['speaker']} for s in rec['segments']]
        write_rttm(ref_rows, ref_path)

        lf.write(f'{uri}\\n')

print('VBx preparation complete. Files in:', vbx_work)

Installing runtime packages...
RUN: /usr/bin/python3 -m pip install -e /content/semester-project/VBx
 scikit-learn->VBx==1.2) (3.6.0)
  Attempting uninstall: VBx
    Found existing installation: VBx 1.2
    Uninstalling VBx-1.2:
      Successfully uninstalled VBx-1.2
  Running setup.py develop for VBx

VBx preparation complete. Files in: artifacts/vbx_work


In [21]:
# Cell 13: Run official VBx (AHC+VB), parse RTTM, and evaluate
def parse_rttm_to_df(rttm_path: Path) -> pd.DataFrame:
    rows = []
    with open(rttm_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip() or line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) < 8 or parts[0] != 'SPEAKER':
                continue
            uri = parts[1]
            st = float(parts[3])
            dur = float(parts[4])
            spk = parts[7]
            rows.append({'uri': uri, 'start': st, 'end': st + dur, 'speaker': spk})
    return pd.DataFrame(rows)

vbx_out_dir = vbx_work / 'out_rttm'
vbx_out_dir.mkdir(parents=True, exist_ok=True)

successful_uris = []
for uri in selected_uris:
    in_list = vbx_work / 'exp' / f'{uri}_list.txt'
    with open(in_list, 'w', encoding='utf-8') as f:
        # VBx predict.py does not strip newline characters from list entries.
        f.write(uri)

    ark_fn = vbx_work / 'exp' / f'{uri}.ark'
    seg_fn = vbx_work / 'exp' / f'{uri}.seg'

    try:
        run_cmd([
            sys.executable, str(VBX_DIR / 'VBx' / 'predict.py'),
            '--in-file-list', str(in_list),
            '--in-lab-dir', str(vbx_work / 'vad'),
            '--in-wav-dir', str(vbx_work / 'audios_16k'),
            '--out-ark-fn', str(ark_fn),
            '--out-seg-fn', str(seg_fn),
            '--weights', str(vbx_model_root / 'nnet' / 'final.onnx'),
            '--backend', 'onnx'
        ])
    except Exception as e:
        print(f'VBx predict failed for {uri}: {e}')
        continue

    if (not seg_fn.exists()) or seg_fn.stat().st_size == 0:
        print(f'VBx predict produced empty segment file for {uri}, skipping VB-HMM for this URI.')
        continue

    try:
        run_cmd([
            sys.executable, str(VBX_DIR / 'VBx' / 'vbhmm.py'),
            '--init', 'AHC+VB',
            '--out-rttm-dir', str(vbx_out_dir),
            '--xvec-ark-file', str(ark_fn),
            '--segments-file', str(seg_fn),
            '--xvec-transform', str(vbx_model_root / 'transform.h5'),
            '--plda-file', str(vbx_model_root / 'plda'),
            '--threshold', '-0.015',
            '--lda-dim', '128',
            '--Fa', '0.3',
            '--Fb', '17',
            '--loopP', '0.99'
        ])
        successful_uris.append(uri)
    except Exception as e:
        print(f'VBx vbhmm failed for {uri}: {e}')

# Aggregate VBx hypotheses
vbx_hyp_parts = []
for uri in successful_uris:
    p = vbx_out_dir / f'{uri}.rttm'
    if p.exists() and p.stat().st_size > 0:
        vbx_hyp_parts.append(parse_rttm_to_df(p))

if vbx_hyp_parts:
    vbx_hyp_df = pd.concat(vbx_hyp_parts, ignore_index=True)
    dev_subset_manifest = dev_manifest[dev_manifest['uri'].isin(successful_uris)].copy()
    vbx_metrics = evaluate_manifest(dev_subset_manifest, vbx_hyp_df, 'DEV SUBSET VBx(AHC+VB)')
else:
    print('No VBx RTTM outputs were produced; falling back to AHC hypothesis for continuity of notebook execution.')
    dev_subset_manifest = dev_manifest[dev_manifest['uri'].isin(selected_uris)].copy()
    vbx_hyp_df = ahc_dev_df[ahc_dev_df['uri'].isin(selected_uris)].copy()
    vbx_metrics = evaluate_manifest(dev_subset_manifest, vbx_hyp_df, 'DEV SUBSET AHC fallback (VBx unavailable)')

print(vbx_metrics)
pd.DataFrame([vbx_metrics]).to_csv(ART / 'metrics' / 'vbx_subset_metrics.csv', index=False)

RUN: /usr/bin/python3 /content/semester-project/VBx/VBx/predict.py --in-file-list artifacts/vbx_work/exp/7879091338544298569_list.txt --in-lab-dir artifacts/vbx_work/vad --in-wav-dir artifacts/vbx_work/audios_16k --out-ark-fn artifacts/vbx_work/exp/7879091338544298569.ark --out-seg-fn artifacts/vbx_work/exp/7879091338544298569.seg --weights /content/semester-project/VBx/VBx/models/ResNet101_16kHz/nnet/final.onnx --backend onnx
RUN: /usr/bin/python3 /content/semester-project/VBx/VBx/vbhmm.py --init AHC+VB --out-rttm-dir artifacts/vbx_work/out_rttm --xvec-ark-file artifacts/vbx_work/exp/7879091338544298569.ark --segments-file artifacts/vbx_work/exp/7879091338544298569.seg --xvec-transform /content/semester-project/VBx/VBx/models/ResNet101_16kHz/transform.h5 --plda-file /content/semester-project/VBx/VBx/models/ResNet101_16kHz/plda --threshold -0.015 --lda-dim 128 --Fa 0.3 --Fb 17 --loopP 0.99
--- STDERR tail ---
#############################################################################

eval:DEV SUBSET AHC fallback (VBx unavailable):   0%|          | 0/1 [00:00<?, ?it/s]

DEV SUBSET AHC fallback (VBx unavailable) DER=1.8477 JER=0.9761
{'title': 'DEV SUBSET AHC fallback (VBx unavailable)', 'DER': 1.8477072099289031, 'JER': 0.9761019103230792}
